<a href="https://colab.research.google.com/github/Bebrochochka/International-Ai-Challenge/blob/main/RosTelekom_AI_Challange_VGG_style.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras import models, layers
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from google.colab import drive
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import RandomOverSampler
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.utils.class_weight import compute_class_weight
from collections import Counter

# === 1. Подключение Google Диска и создание ярлыка на папку с изображениями ===
#drive.mount('/content/drive')
#!ln -s "/content/drive/MyDrive/images" images

# === 2. Загрузка и обработка обучающего датасета ===
data = pd.read_csv("train_r2.csv")
X_files = data["image"]
y = data["class"]

img = []
for i in X_files:
    imgag = Image.open(f"images/{i}").convert("RGB")
    imgag = imgag.resize((360, 360))
    imgag = np.array(imgag)
    img.append(imgag)

X = np.array(img) / 255.0  # нормализация
y = np.array(y)

plt.imshow(X[0])  # показать первое изображение

for i in range(1):
  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=102)

  # === 4. Балансировка train через RandomOverSampler ===
  print("До балансировки:", Counter(y_train))

  X_train_flat = X_train.reshape(len(X_train), -1)
  ros = RandomOverSampler(random_state=42)
  X_train_resampled, y_train_resampled = ros.fit_resample(X_train_flat, y_train)
  X_train = X_train_resampled.reshape(-1, 360, 360, 3)
  y_train = y_train_resampled

  print("После ROS:", Counter(y_train))

  # === 5. Обработка реального теста ===
  realTestData = pd.read_csv("test_r2.csv")
  realTest = realTestData["image"]
  testdataID = realTestData['image']
  imgg = []
  for i in realTest:
      imgag = Image.open(f"images/{i}").convert("RGB")
      imgag = imgag.resize((360, 360))
      imgag = np.array(imgag)
      imgg.append(imgag)

  realTest = np.array(imgg) / 255.0
  #plt.imshow(realTest[0])  # показать первое изображение из теста

  # === 6. Построение модели Vgg-Style.CNN ===
  cnn = models.Sequential([
      layers.Input(shape=(360, 360, 3)),
      layers.Conv2D(filters=60, kernel_size=(3, 3), activation="relu"),
      layers.Conv2D(filters=40, kernel_size=(3, 3), activation="relu"),
      layers.MaxPooling2D(2, 2),

      layers.Conv2D(filters=60, kernel_size=(3, 3), activation="relu"),
      layers.Conv2D(filters=40, kernel_size=(3, 3), activation="relu"),
      layers.MaxPooling2D(2, 2),

      layers.Flatten(),
      layers.Dense(44, activation="swish"),
      layers.Dropout(0.6),
      layers.Dense(24, activation="swish"),
      layers.Dropout(0.6),
      layers.Dense(34, activation="swish"),
      layers.Dense(1, activation="sigmoid"),
  ])

  cnn.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

  # === 7. Callbacks и веса классов ===
  earlystop = EarlyStopping(monitor='val_loss', patience=50, restore_best_weights=True)


  # === 8. Обучение модели ===
  history = cnn.fit(
      X_train, y_train,
      epochs=55,
      batch_size=32,
      validation_split=0.2,
      callbacks=[earlystop],
      class_weight={0: 3.0, 1: 1.5}
  )

  # === 9. Предсказание и оценка ===
  y_hat = cnn.predict(X_test)
  y_hat = [1 if val >= 0.5 else 0 for val in y_hat]

  #output = pd.DataFrame({
  #     "image": testdataID,
  #     "class": y_hat
  #})
#
  #output.to_csv("submission.csv", index=False)

  report = classification_report(y_test, y_hat, output_dict=False)
  print(report)
  print(confusion_matrix(y_test, y_hat))

  # Loss
  plt.figure(figsize=(12, 5))
  plt.subplot(1, 2, 1)
  plt.plot(history.history['loss'], label='Train Loss')
  plt.plot(history.history['val_loss'], label='Validation Loss')
  plt.xlabel('Epoch')
  plt.ylabel('Loss')
  plt.title('Training and Validation Loss')
  plt.legend()

  # Accuracy
  plt.subplot(1, 2, 2)
  plt.plot(history.history['accuracy'], label='Train Accuracy')
  plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
  plt.xlabel('Epoch')
  plt.ylabel('Accuracy')
  plt.title('Training and Validation Accuracy')
  plt.legend()

  plt.tight_layout()
  plt.show()